# Virelion-DCCP — One-Click Runtime Bug Hunt

This notebook is deliberately self-contained. **Every test runs in a child Python process with an explicit `PYTHONPATH`**, so notebook kernel state, cell order, cached imports, and stale `sys.path` cannot affect the result.

It does **not** upgrade or replace Colab's global `pip`, `setuptools`, Torch, or other installed packages. Validation dependencies are installed into `/content/dccp-runtime-deps` with `pip --target`.

**Do not use an older DCCP validation notebook. Run this notebook from Cell 1.**


In [ ]:
# 1. Fresh checkout + isolated dependency directory
from pathlib import Path
import os, shutil, subprocess, sys

ROOT = Path('/content/Virelion-DCCP-RUNTIME-HUNT')
DEPS = Path('/content/dccp-runtime-deps')

for p in (ROOT, DEPS):
    if p.exists():
        shutil.rmtree(p)

subprocess.run([
    'git','clone','--branch','main','--depth','1',
    'https://github.com/Virelion-Biotech/Virelion-DCCP.git',str(ROOT)
], check=True)

DEPS.mkdir(parents=True)
subprocess.run([
    sys.executable,'-m','pip','install',
    '--disable-pip-version-check','--no-warn-script-location',
    '--target',str(DEPS),
    'jsonschema','pytest','pytest-cov','coverage','hypothesis','pip-audit','ruff==0.16.7'
], check=True)

COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
print('TEST COMMIT:', COMMIT)
print('ROOT:', ROOT)
print('DEPS:', DEPS)


In [ ]:
# 2. ONE-CLICK RUNTIME BUG HUNT
# Everything below executes in fresh child processes with explicit PYTHONPATH.
ENV = os.environ.copy()
ENV['PYTHONPATH'] = os.pathsep.join([str(DEPS), str(ROOT/'src')])
ENV['PYTHONNOUSERSITE'] = '1'

RUNNER = ROOT/'_colab_runtime_hunt.py'
RUNNER.write_text(r'''\
from pathlib import Path
import json, math, subprocess, sys, tempfile
        
ROOT = Path.cwd()
SCENARIOS = ROOT/'scenarios'
SRC = ROOT/'src'
results = []
        
def record(name, ok, detail=''):
    results.append({'test': name, 'ok': bool(ok), 'detail': detail})
    print(('PASS' if ok else 'FAIL') + ' :: ' + name)
    if detail:
        print(detail)
        
# Environment / import identity
try:
    import dccp
    record('import dccp', str(SRC) in str(dccp.__file__), str(dccp.__file__))
except Exception as exc:
    record('import dccp', False, repr(exc))
        
# Every module import
for path in sorted((SRC/'dccp').glob('*.py')):
    if path.name == '__init__.py':
        continue
    name = path.stem
    try:
        __import__(f'dccp.{name}')
        record(f'import dccp.{name}', True)
    except Exception as exc:
        record(f'import dccp.{name}', False, repr(exc))
        
# Individual scenario load + audit
try:
    from dccp.scenario import load_scenario
    from dccp.audit import audit_scenario
    files = sorted(SCENARIOS.rglob('*.json'))
    record('scenario discovery', bool(files), f'count={len(files)}')
    for path in files:
        try:
            sc = load_scenario(path)
            ar = audit_scenario(sc)
            detail = '' if ar.passed else repr({'schema': ar.schema_errors, 'policy': ar.policy_errors})
            record(f'audit {path.relative_to(ROOT)}', ar.passed, detail)
        except Exception as exc:
            record(f'audit {path.relative_to(ROOT)}', False, repr(exc))
except Exception as exc:
    record('scenario API import', False, repr(exc))
        
# Registry/library/challenge/bundle integration
try:
    from dccp.registry import build_registry, write_registry
    from dccp.library import discover_scenarios, load_library, materialize_challenge_set, write_challenge_set
    from dccp.bundle import build_bundle
    with tempfile.TemporaryDirectory() as td:
        tmp = Path(td)
        files = discover_scenarios(SCENARIOS)
        reg = build_registry(SCENARIOS, exclude_paths=[tmp/'registry.json'])
        reg_payload = write_registry(SCENARIOS, tmp/'registry.json')
        assert len(reg) == len(files) == reg_payload['n_entries']
        lib = load_library(SCENARIOS)
        assert len(lib) == len(files)
        challenge = materialize_challenge_set(SCENARIOS)
        assert challenge['n_cases'] == len(lib) and len(challenge['set_hash']) == 64
        write_challenge_set(tmp/'challenge.json', SCENARIOS)
        written = json.loads((tmp/'challenge.json').read_text())
        assert written['set_hash'] == challenge['set_hash']
        bundle = build_bundle(tmp/'bundle', run_id='colab-runtime', input_files=[files[0],files[0]], producer_version='0.3.0', base_dir=ROOT)
        assert len(bundle['inputs']) == 1 and not bundle['inputs'][0]['path'].startswith('/')
        record('registry/library/challenge/bundle integration', True, f'files={len(files)}')
except Exception as exc:
    record('registry/library/challenge/bundle integration', False, repr(exc))
        
# Edge/path safety
try:
    from dccp.fingerprint import canonical_json
    from dccp.bundle import build_bundle
    for value in (float('nan'), float('inf'), float('-inf')):
        try:
            canonical_json({'x': value})
            raise AssertionError(f'accepted {value}')
        except (ValueError, TypeError):
            pass
    with tempfile.TemporaryDirectory() as td:
        base = Path(td)/'base'; base.mkdir()
        outside = Path(td)/'outside.txt'; outside.write_text('x')
        try:
            build_bundle(base/'bundle', run_id='outside', input_files=[outside], base_dir=base)
            raise AssertionError('outside file was accepted')
        except (ValueError, FileNotFoundError, RuntimeError):
            pass
    record('numeric/path safety edge cases', True)
except Exception as exc:
    record('numeric/path safety edge cases', False, repr(exc))
        
# CLI: test every CLI command used by CI
cli_commands = [
    ['-m','dccp.cli','--version'],
    ['-m','dccp.cli','--help'],
    ['-m','dccp.cli','validate','scenarios/examples/SCENARIO-001.ordinary-mi.json'],
    ['-m','dccp.cli','audit-all','scenarios/examples'],
    ['-m','dccp.cli','registry','--root','scenarios/examples','--output','/tmp/dccp-registry.json'],
    ['-m','dccp.cli','release-gate','/tmp/dccp-registry.json','--base','.'],
    ['-m','dccp.cli','materialize','--root','scenarios/examples','--output','/tmp/dccp-challenge-set.json'],
    ['-m','dccp.cli','map-scores','--scores','inflammatory=0.8,contractile_functional=0.4','--draft-id','SCENARIO-902','-o','/tmp/draft.json'],
    ['-m','dccp.cli','validate','/tmp/draft.json'],
    ['-m','dccp.cli','bundle','--output-dir','/tmp/dccp-bundle','--run-id','ci','--input-files','scenarios/examples/SCENARIO-001.ordinary-mi.json'],
]
for cmd in cli_commands:
    try:
        p = subprocess.run([sys.executable, *cmd], text=True, capture_output=True)
        detail = (p.stdout + '\n' + p.stderr).strip()[-4000:]
        record('CLI ' + ' '.join(cmd), p.returncode == 0, '' if p.returncode == 0 else detail)
    except Exception as exc:
        record('CLI ' + ' '.join(cmd), False, repr(exc))
        
# Ruff exact CI version
p = subprocess.run([sys.executable,'-m','ruff','check','src','tests'], text=True, capture_output=True)
record('ruff 0.16.7 check src tests', p.returncode == 0, '' if p.returncode == 0 else (p.stdout+p.stderr)[-6000:])
        
# Actual pytest suite
p = subprocess.run([sys.executable,'-m','pytest','-q'], text=True, capture_output=True)
record('pytest -q', p.returncode == 0, '' if p.returncode == 0 else (p.stdout+p.stderr)[-6000:])
        
# Determinism / repeated execution
try:
    from dccp.library import load_library
    from dccp.provenance import canonical_hash
    seen=[]
    for _ in range(25):
        lib=load_library(SCENARIOS)
        seen.append(canonical_hash({'ids':sorted(x.scenario.scenario_id for x in lib),'digests':sorted(x.digest for x in lib)}))
    record('25 repeated library loads deterministic', len(set(seen)) == 1)
except Exception as exc:
    record('25 repeated library loads deterministic', False, repr(exc))
        
# Final machine-readable result
payload = {
    'commit': subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),
    'total_tests': len(results),
    'passed': sum(x['ok'] for x in results),
    'failed': sum(not x['ok'] for x in results),
    'results': results,
}
(ROOT/'colab_runtime_results.json').write_text(json.dumps(payload, indent=2) + '\n')
print('\n=== FINAL RESULT ===')
print(json.dumps({k:payload[k] for k in ('commit','total_tests','passed','failed')}, indent=2))
if payload['failed']:
    print('\nFAILED TESTS:')
    for x in results:
        if not x['ok']:
            print(x['test'])
            if x['detail']: print(x['detail'])
''', encoding='utf-8')

p = subprocess.run([sys.executable, str(RUNNER)], cwd=ROOT, env=ENV, text=True, capture_output=True)
print(p.stdout)
if p.stderr:
    print('STDERR:\n' + p.stderr)
print('Runner exit code:', p.returncode)
assert p.returncode == 0 or 'FAILED TESTS:' in p.stdout


In [ ]:
# 3. Inspect the result without relying on notebook variables.
import json
from pathlib import Path

report = json.loads((Path('/content/Virelion-DCCP-RUNTIME-HUNT')/'colab_runtime_results.json').read_text())
print(json.dumps({
    'commit': report['commit'],
    'total_tests': report['total_tests'],
    'passed': report['passed'],
    'failed': report['failed'],
}, indent=2))

if report['failed']:
    print('\nFAILURES:')
    for item in report['results']:
        if not item['ok']:
            print('\nTEST:', item['test'])
            print(item['detail'])
else:
    print('\nALL RUNTIME TESTS PASSED.')


In [ ]:
# 4. Optional: expose the JSON report for download from Colab.
from google.colab import files
files.download('/content/Virelion-DCCP-RUNTIME-HUNT/colab_runtime_results.json')
